4週間分の，6:00-10:00，16:00-22:00の利用客数を出力する


In [21]:
import polars as pl
import pandas as pd # 念の為importしておく
import matplotlib.pyplot as plt
import japanize_matplotlib 
from collections import defaultdict

In [22]:
df_202504 = pl.read_csv('/home/takizawa/2025_takizawa/2026_Competition/202504_od_data.csv')
df_202507 = pl.read_csv('/home/takizawa/2025_takizawa/2026_Competition/202507_od_data.csv')
df_202510 = pl.read_csv('/home/takizawa/2025_takizawa/2026_Competition/202510_od_data.csv')
df_202601 = pl.read_csv('/home/takizawa/2025_takizawa/2026_Competition/202601_od_data.csv')

In [23]:
# df_202504.head()

In [24]:
def get_peak_time_dropoff_count(df, target_date):
    """
    指定した乗車日に対して、6:00-10:00 および 16:00-22:00 の
    降車客数をカウントして返す関数。
    
    Parameters
    ----------
    df : polars.DataFrame
        対象のデータフレーム
    target_date : int
        抽出対象の乗車日。例: 20250407
        
    Returns
    -------
    tuple(int, int)
        (6:00-10:00の利用者数, 16:00-22:00の利用者数)
    """

    # 対象日のデータだけ抽出
    df_target = df.filter(
        pl.col("乗車日") == target_date
    )

    # 「降車時刻」列をリストとして取り出す
    dropoff_times = df_target["降車時刻"].to_list()

    # カウント用変数
    count_morning = 0  # 6:00-10:00
    count_evening = 0  # 16:00-22:00

    # 1つずつ降車時刻を処理
    for time_str in dropoff_times:
        # 欠損値を除外
        if time_str is None:
            continue

        # 念のため文字列に変換
        time_str = str(time_str)

        # 空文字や不正な値を除外
        if ":" not in time_str:
            continue

        # ":" の前後で分割してint型に変換
        forward, backward = time_str.split(":")
        
        # 時刻を「0時からの経過分」に変換
        total_minutes = int(forward) * 60 + int(backward)

        # 6:00 (360分) 〜 10:00 (600分未満) の判定
        # ※10:00ちょうどを含める場合は `< 600` を `<= 600` に変更してください
        if 360 <= total_minutes < 600:
            count_morning += 1
            
        # 16:00 (960分) 〜 22:00 (1320分未満) の判定
        # ※22:00ちょうどを含める場合は `< 1320` を `<= 1320` に変更してください
        elif 960 <= total_minutes < 1320:
            count_evening += 1

    return count_morning, count_evening

In [27]:
from datetime import datetime
from collections import defaultdict

# 処理対象のデータフレームと日付範囲
target_configs = [
    (df_202504, range(20250414, 20250421)),
    (df_202507, range(20250707, 20250714)),
    (df_202510, range(20251006, 20251013)),
    (df_202601, range(20260119, 20260126))
]

# 抽出したい駅のリストと、出力用の曜日リスト
target_stations = ["泉ケ丘", "難波", "新今宮"]
weekdays_str = ["月", "火", "水", "木", "金", "土", "日"]

# データを蓄積するための辞書
# 構造: station_stats[駅名][曜日インデックス(0~6)] = {'morning': [人数, ...], 'evening': [人数, ...]}
station_stats = defaultdict(lambda: defaultdict(lambda: {'morning': [], 'evening': []}))

# ==========================================
# 1. 各日程のデータを計算し、曜日ごとに振り分けて蓄積
# ==========================================
for df, dates in target_configs:
    for station in target_stations:
        # 駅名で事前にフィルタリング
        df_station = df.filter(pl.col("降車駅") == station)
        
        for date in dates:
            morning_cnt, evening_cnt = get_peak_time_dropoff_count(
                df=df_station,
                target_date=date
            )
            
            # 日付文字列から曜日を取得 (0: 月曜, 1: 火曜, ..., 6: 日曜)
            dt = datetime.strptime(str(date), "%Y%m%d")
            weekday_idx = dt.weekday()
            
            # 利用客数が0の場合は除外（0より大きい場合のみリストに追加）
            if morning_cnt > 0:
                station_stats[station][weekday_idx]['morning'].append(morning_cnt)
            if evening_cnt > 0:
                station_stats[station][weekday_idx]['evening'].append(evening_cnt)

# ==========================================
# 2. 蓄積したリストから曜日ごとの平均値を計算して出力
# ==========================================
print("=== 曜日ごとの平均利用客数（利用客数0の日を除く） ===")
for station in target_stations:
    print(f"\n■ {station}駅")
    
    for i, weekday in enumerate(weekdays_str):
        morning_list = station_stats[station][i]['morning']
        evening_list = station_stats[station][i]['evening']
        
        # リストに要素がある（0より大きい日が存在する）場合のみ平均を計算。ない場合は0.0とする。
        morning_avg = sum(morning_list) / len(morning_list) if morning_list else 0.0
        evening_avg = sum(evening_list) / len(evening_list) if evening_list else 0.0
        
        print(f"{weekday}曜日 | 6:00-10:00: {morning_avg:6.1f} 人 | 16:00-22:00: {evening_avg:6.1f} 人")

=== 曜日ごとの平均利用客数（利用客数0の日を除く） ===

■ 泉ケ丘駅
月曜日 | 6:00-10:00: 4323.8 人 | 16:00-22:00: 7846.8 人
火曜日 | 6:00-10:00: 4435.8 人 | 16:00-22:00: 7787.8 人
水曜日 | 6:00-10:00: 4463.7 人 | 16:00-22:00: 7770.0 人
木曜日 | 6:00-10:00: 4325.7 人 | 16:00-22:00: 7727.0 人
金曜日 | 6:00-10:00: 4437.8 人 | 16:00-22:00: 7618.0 人
土曜日 | 6:00-10:00: 2964.0 人 | 16:00-22:00: 4582.2 人
日曜日 | 6:00-10:00: 1701.5 人 | 16:00-22:00: 3849.2 人

■ 難波駅
月曜日 | 6:00-10:00: 43842.0 人 | 16:00-22:00: 20415.8 人
火曜日 | 6:00-10:00: 43369.8 人 | 16:00-22:00: 20544.5 人
水曜日 | 6:00-10:00: 42178.5 人 | 16:00-22:00: 21002.8 人
木曜日 | 6:00-10:00: 43219.2 人 | 16:00-22:00: 21205.8 人
金曜日 | 6:00-10:00: 42747.2 人 | 16:00-22:00: 26132.8 人
土曜日 | 6:00-10:00: 19127.0 人 | 16:00-22:00: 21690.5 人
日曜日 | 6:00-10:00: 15048.5 人 | 16:00-22:00: 17901.8 人

■ 新今宮駅
月曜日 | 6:00-10:00: 20849.8 人 | 16:00-22:00: 12369.8 人
火曜日 | 6:00-10:00: 20894.2 人 | 16:00-22:00: 12308.8 人
水曜日 | 6:00-10:00: 20591.2 人 | 16:00-22:00: 12809.5 人
木曜日 | 6:00-10:00: 20650.2 人 | 16:00-22:00: 12549.5 人
金曜日 |